In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import backtrader as bt

/Users/dipalshah/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [16]:
stocks = ["AAPL", "MSFT", "GOOG", "NVDA", "META", "AMZN", "TSLA"]
df = yf.download(stocks, group_by = "tickers")

[*********************100%%**********************]  7 of 7 completed


2 Failed downloads:
['TSLA', 'GOOG']: Exception('%ticker%: No price data found, symbol may be delisted (1d 1926-03-08 -> 2025-02-11)')


In [6]:
df = df.dropna()

In [7]:
df.to_csv("data.csv")

In [17]:
df = pd.read_csv("data.csv", index_col = 0, header = [0, 1])

In [18]:
df

META                                                  \
                  Open        High         Low       Close   Adj Close   
Date                                                                     
2012-05-18   42.049999   45.000000   38.000000   38.230000   38.084518   
2012-05-21   36.529999   36.660000   33.000000   34.029999   33.900505   
2012-05-22   32.610001   33.590000   30.940001   31.000000   30.882032   
2012-05-23   31.370001   32.500000   31.360001   32.000000   31.878227   
2012-05-24   32.950001   33.209999   31.770000   33.029999   32.904308   
...                ...         ...         ...         ...         ...   
2025-02-05  703.549988  718.140015  699.010010  704.869995  704.869995   
2025-02-06  705.880005  718.900024  703.500000  711.989990  711.989990   
2025-02-07  716.799988  725.010010  711.750000  714.520020  714.520020   
2025-02-10  718.559998  721.200012  711.330017  717.400024  717.400024   
2025-02-11  713.320007  723.659973  710.039978  719.799988  719.799988   

                               AMZN                                      ...  \
                 Volume        Open        High         Low       Close  ...   
Date                                                                     ...   
2012-05-18  573576400.0   10.970500   10.981500   10.640500   10.692500  ...   
2012-05-21  168192700.0   10.701500   10.999000   10.641000   10.905500  ...   
2012-05-22  101786600.0   10.915500   10.943500   10.698000   10.766500  ...   
2012-05-23   73600000.0   10.735500   10.877500   10.559000   10.864000  ...   
2012-05-24   50237200.0   10.849000   10.883000   10.635000   10.762000  ...   
...                 ...         ...         ...         ...         ...  ...   
2025-02-05   17778200.0  237.020004  238.320007  235.199997  236.169998  ...   
2025-02-06   13080700.0  238.009995  239.660004  236.009995  238.830002  ...   
2025-02-07   16427100.0  232.500000  234.809998  228.059998  229.149994  ...   
2025-02-10   12904300.0  230.550003  233.919998  229.199997  233.139999  ...   
2025-02-11   11880595.0  231.914993  233.440002  230.130005  232.759995  ...   

                  GOOG                                             TSLA  \
                   Low       Close   Adj Close       Volume        Open   
Date                                                                      
2012-05-18   14.861794   14.953949   14.900410  239835606.0    1.891333   
2012-05-21   14.943986   15.295419   15.240657  123477094.0    1.838667   
2012-05-22   14.844360   14.963912   14.910338  122533571.0    2.006667   
2012-05-23   14.872255   15.179603   15.125257  127600492.0    2.037333   
2012-05-24   14.915842   15.035145   14.981316   75935562.0    2.083333   
...                ...         ...         ...          ...         ...   
2025-02-05  189.910004  193.300003  193.300003   43666400.0  387.510010   
2025-02-06  190.490005  193.309998  193.309998   20816600.0  373.029999   
2025-02-07  185.100006  187.139999  187.139999   29565700.0  370.190002   
2025-02-10  187.610001  188.199997  188.199997   16606000.0  356.209991   
2025-02-11  186.080002  187.070007  187.070007   12937988.0  345.825012   

                                                                         
                  High         Low       Close   Adj Close       Volume  
Date                                                                     
2012-05-18    1.897333    1.788667    1.837333    1.837333   24247500.0  
2012-05-21    1.950667    1.808000    1.918000    1.918000   22128000.0  
2012-05-22    2.089333    2.000000    2.053333    2.053333   35493000.0  
2012-05-23    2.070000    1.966667    2.068000    2.068000   18306000.0  
2012-05-24    2.083333    1.979333    2.018667    2.018667   16134000.0  
...                ...         ...         ...         ...          ...  
2025-02-05  388.390015  375.529999  378.170013  378.170013   57223300.0  
2025-02-06  375.399994  363.179993  374.320007  374.320007   77918200

In [7]:
weights = pd.DataFrame({
            "date": pd.date_range(start=df.index[0], periods=len(df.index), freq='D'),
            "AAPL": [.1] * len(df.index), 
            "MSFT": [.1] * len(df.index), 
            "GOOG": [.1] * len(df.index), 
            "NVDA": [.1] * len(df.index), 
            "META": [.1] * len(df.index), 
            "AMZN": [.1] * len(df.index), 
            "TSLA": [.1] * len(df.index)
        }).set_index('date')

In [8]:
class Trader(bt.Strategy):
    def __init__(self):
        self.weights = weights

    def next(self):
        dt = self.datas[0].datetime.date(0)
        if dt in self.weights.index:
            weight_AAPL = self.weights.loc[dt, "AAPL"]
            weight_MSFT = self.weights.loc[dt, "MSFT"]
            weight_GOOG = self.weights.loc[dt, "GOOG"]
            weight_NVDA = self.weights.loc[dt, "NVDA"]
            weight_META = self.weights.loc[dt, "META"]
            weight_AMZN = self.weights.loc[dt, "AMZN"]
            weight_TSLA = self.weights.loc[dt, "TSLA"]
            
            self.order_target_percent(data = self.datas[0], target = weight_AAPL)
            self.order_target_percent(data = self.datas[1], target = weight_MSFT)
            self.order_target_percent(data = self.datas[2], target = weight_GOOG)
            self.order_target_percent(data = self.datas[3], target = weight_NVDA)
            self.order_target_percent(data = self.datas[4], target = weight_META)
            self.order_target_percent(data = self.datas[5], target = weight_AMZN)
            self.order_target_percent(data = self.datas[6], target = weight_TSLA)

In [9]:
data_feeds = []

for stock in stocks:
    data = bt.feeds.PandasData(dataname = df[stock])
    data_feeds.append(data)

In [10]:
cerebro = bt.Cerebro()
cerebro.addstrategy(Trader)

for data in data_feeds:
    cerebro.adddata(data)

cerebro.broker.set_cash(10_000)
cerebro.run()

In [11]:
cerebro.plot()

<IPython.core.display.Javascript object>